# Chapter 25: Reinsurance Basics

Code of Chapter 25 of *Practical Introduction to Non-Life Insurance Actuarial Mathematics* (F.-X. Cart-Tanneur). Run the cells from the top: later cells reuse the data and imports of earlier ones. Each comment after `#` gives the value printed in the book.

In [ ]:
# Setup: make the book's modules importable (works locally and in the browser)
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

## Exercise 3: A 10-10 test by hand

In [ ]:
import numpy as np
from gireins import risk_transfer
L = np.array([0] * 70 + [50] * 20 + [150] * 8 + [400] * 2, float)
risk_transfer(L, 60)                              # p_loss_10 0.10, erd 0.233

## 25.7 Python Walkthrough

In [ ]:
import numpy as np
from gitriangle import synthetic_history
from gilossmodels import fit_mle
from gireins import layer, quota_share, surplus_share, simulate_layers, risk_transfer

# Sections 25.1.3 and 25.1.4: hand examples
x = np.array([20_000, 80_000, 180_000, 400_000])
quota_share(x, 0.4)[0].sum(), layer(x, 100_000, 150_000).sum()     # 272000, 230000
surplus_share([80_000, 300_000, 800_000], 100_000, 4)              # 0, 0.667, 0.5

# Section 25.3: the 2025 claims of the synthetic portfolio
policies, claims, tx = synthetic_history()
x25 = claims[claims.accident_date.dt.year == 2025].ultimate_amount.to_numpy()
quota_share(x25, 0.3)[0].sum(), layer(x25, 25_000, 25_000).sum()   # 997693, 270543

L = fit_mle(x25, "lognormal")
mu, s = float(L["mu"]), float(L["sigma"])
sim = simulate_layers(654, lambda rng, n: rng.lognormal(mu, s, n), [(25_000, 25_000)], 20_000, seed=25)
xl = sim["25,000 xs 25,000"]
net = sim["gross"] - xl
np.quantile(net, 0.995) - net.mean()                               # 669982

# Section 25.4.3: risk transfer
P = xl.mean() / 0.8
risk_transfer(xl, P)                                               # p_loss_10 0.105, erd 0.029
risk_transfer(np.minimum(xl, 1.05 * P), P)                         # p_loss_10 0.0, erd 0.008